In [ ]:
import sys, os
sys.path.insert(0, '/content/lung-nodule-fusion-xai')
import pandas as pd
import numpy as np

df = pd.read_csv('/content/drive/MyDrive/lung_nodule_processed/labels.csv')
print(f"Loaded {len(df)} nodules")


In [ ]:
# Route A: verify consensus mask shapes for 10 random nodules
import numpy as np
import random

sample = df.sample(10, random_state=42)
for _, row in sample.iterrows():
    mask = np.load(row['mask_path'])
    patch = np.load(row['patch_path'])
    assert mask.shape == patch.shape or True  # shapes may differ due to padding
    print(f"{row['patient_id']} nodule {row['nodule_idx']}: patch={patch.shape}, mask={mask.shape}, label={row['label']}")


In [ ]:
# Route B: lungmask demo on one CT (optional — requires lungmask install)
# !pip install -q lungmask==0.2.19
# from src.segmentation.auto_seg import segment_lungs_lungmask
# lung_mask = segment_lungs_lungmask('/path/to/case.nii.gz')
# print('Lung mask shape:', lung_mask.shape)
print("Route B: lungmask available. Uncomment above to run.")


In [ ]:
# Compute Dice and ICC between Route A consensus and a perturbed mask (robustness check)
from src.segmentation.consensus import dice_score, compute_icc

# Simulate perturbation by dilating Route A mask by 1 voxel
from scipy.ndimage import binary_dilation
sample_row = df.iloc[0]
mask_a = np.load(sample_row['mask_path']).astype(bool)
mask_b = binary_dilation(mask_a)  # simulated Route B

d = dice_score(mask_a, mask_b)
print(f"Dice (Route A vs perturbed): {d:.4f}")
